# Section 3 · Notebook 2 — LangGraph + AgentCore Memory (short-term memory) (Groq edition)

> **Groq edition.** Same lesson as `math_agent_with_checkpointing.ipynb`; only the agent's model runs on Groq. The checkpointer is still Amazon Bedrock AgentCore Memory, so you need AWS credentials — but no Bedrock model access. Set `GROQ_API_KEY` in your environment or a `.env` file.

We give a LangGraph agent **session memory** by using Amazon Bedrock AgentCore Memory as its *checkpointer*. Every step of the conversation is saved automatically, so the agent can pick up where it left off — even from a different process or machine.

| | |
|:--|:--|
| Use case | Multi-step math calculations |
| Framework | LangGraph (via `langchain.agents.create_agent`) |
| Memory | AgentCore Memory — `AgentCoreMemorySaver` checkpointer |
| Complexity | Beginner |

```
 user ──► LangGraph agent ──► Groq model
               │  ▲
     save state│  │load state  (every step)
               ▼  │
        AgentCore Memory  (session_id = thread_id, actor_id)
```

**Prerequisites:** a `GROQ_API_KEY`, AWS credentials, and IAM permissions for AgentCore Memory (`bedrock-agentcore:CreateMemory`, `CreateEvent`, `ListEvents`, `GetMemory`, `DeleteMemory`, ...).

In [ ]:
%pip install -q -r requirements-groq.txt

> If pip upgraded any packages, restart the kernel before continuing.

In [ ]:
import os

from bedrock_agentcore.memory import MemoryClient
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langgraph_checkpoint_aws import AgentCoreMemorySaver

load_dotenv()  # reads GROQ_API_KEY from a .env file, if present

MODEL_ID = os.getenv("GROQ_MODEL_ID", "openai/gpt-oss-120b")
# AgentCore Memory lives in an AWS Region
REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
print(f"Model: {MODEL_ID} | Region: {REGION}")

### Create (or reuse) the AgentCore Memory resource

`create_or_get_memory` creates the memory the first time and waits until it is ACTIVE. Running the notebook again simply reuses it. No extraction strategies are needed for short-term memory.

In [ ]:
client = MemoryClient(region_name=REGION)
memory = client.create_or_get_memory(
    name="MathLanggraphAgentGroq",
    description="Short-term memory for the LangGraph math agent",
)
memory_id = memory["id"]
print("Memory ID:", memory_id, "| status:", memory.get("status"))

### Configure the checkpointer and the model

- `memory_id` — the AgentCore Memory resource that stores checkpoints
- `region_name` — where it lives
- `MODEL_ID` — the Groq model that powers the agent

In [ ]:
checkpointer = AgentCoreMemorySaver(memory_id, region_name=REGION)
llm = init_chat_model(MODEL_ID, model_provider="groq", max_retries=5)  # reads GROQ_API_KEY

### Math tools

In [ ]:
@tool
def add(a: int, b: int) -> int:
    """Add two integers and return the result."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers and return the result."""
    return a * b


tools = [add, multiply]

### Build the agent with the AgentCore Memory checkpointer

In [ ]:
graph = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Use the tools for every calculation.",
    checkpointer=checkpointer,
)
graph

### Run it

The checkpointer needs two IDs in the run config:
- `thread_id` → AgentCore **session_id** (one conversation)
- `actor_id` → AgentCore **actor_id** (the user or agent)

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-1",  # REQUIRED: maps to AgentCore session_id
        "actor_id": "react-agent-1",  # REQUIRED: maps to AgentCore actor_id
    }
}

inputs = {"messages": [{"role": "user", "content": "What is 1337 times 515321? Then add 412 and return the value to me."}]}

for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### Inspect the saved state

In [ ]:
for message in graph.get_state(config).values.get("messages"):
    print(f"{message.type}: {message.text}")
    print("=" * 40)

### Checkpoint history (most recent first)

In [ ]:
for checkpoint in graph.get_state_history(config):
    print(
        f"Checkpoint {checkpoint.config['configurable']['checkpoint_id']} "
        f"— messages in state: {len(checkpoint.values.get('messages', []))}"
    )

### Memory persistence — continue the same session

In [ ]:
inputs = {"messages": [{"role": "user", "content": "What were the first calculations I asked you to do?"}]}

for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

### Session isolation — a new session starts empty

In [ ]:
config = {
    "configurable": {
        "thread_id": "session-2",  # new session
        "actor_id": "react-agent-1",  # same actor
    }
}

inputs = {"messages": [{"role": "user", "content": "What values did I ask you to multiply and add?"}]}
for chunk in graph.stream(inputs, stream_mode="updates", config=config):
    print(chunk)

## Summary

1. Created an AgentCore Memory resource
2. Used `AgentCoreMemorySaver` as the LangGraph checkpointer
3. Saw state persist across turns in the same session
4. Saw a new session start with a clean slate

### Clean up

In [ ]:
client.delete_memory_and_wait(memory_id=memory_id, max_wait=300, poll_interval=10)
print("Deleted memory", memory_id)